# 4.5 Übung: Dachstuhl-Fachwerk (sechs Knoten)

In Kapitel 4.4 haben Sie das FEM-Schema an einem Fünfknoten-Fachwerk mit
detaillierter Schritt-für-Schritt-Anleitung erprobt. In dieser Übung wenden
Sie denselben Algorithmus auf ein neues Tragwerk an. Die Unterstützung ist
bewusst geringer: Die Code-Zellen enthalten keine Kommentare mehr, die jeden
Teilschritt vorschreiben. Die ausklappbaren Lösungsblöcke stehen weiterhin
zur Verfügung, sollten aber erst nach einem eigenen Lösungsversuch geöffnet
werden.

## Lernziele

* [ ] Sie können das FEM-Berechnungsschema für ebene Fachwerke selbstständig
  auf ein neues Tragwerk anwenden, ohne Schritt-für-Schritt-Anleitung.
* [ ] Sie können Verschiebungen, Lagerkräfte und Stabkräfte berechnen und
  die Ergebnisse auf physikalische Plausibilität prüfen.
* [ ] Sie können den Einfluss von Parametern (Querschnitt, Laststellung)
  numerisch untersuchen und das Ergebnis mit dem Zusammenhang $k = EA/L$
  begründen.

## Problemstellung

Betrachten Sie ein ebenes Fachwerk mit sechs Knoten und neun Stäben, das
einem einfachen Dachstuhl nachempfunden ist. Die vier Untergurtknoten liegen
auf der Linie $y = 0\,\text{m}$, die beiden Obergurtknoten auf
$y = 1\,\text{m}$:

| Knoten | $x$ in m | $y$ in m | Bemerkung |
| --- | --- | --- | --- |
| 0 | 0.0 | 0.0 | Festlager |
| 1 | 1.0 | 0.0 | frei |
| 2 | 2.0 | 0.0 | frei |
| 3 | 3.0 | 0.0 | Festlager |
| 4 | 1.0 | 1.0 | frei |
| 5 | 2.0 | 1.0 | frei |

Die neun Stäbe verbinden folgende Knotenpaare:

| Gruppe | Stäbe |
| --- | --- |
| Untergurt | 0-1, 1-2, 2-3 |
| Obergurt | 4-5 |
| Randdiagonalen | 0-4, 3-5 |
| Innenstreben | 1-4, 2-5 |
| Kreuzdiagonale | 1-5 |

An den Knoten 4 und 5 greift je eine Vertikallast von $3000\,\text{N}$ nach
unten an. Die Materialeigenschaften sind dieselben wie in den Kapiteln 4.1
bis 4.4: Stahl mit $E = 2{,}1 \cdot 10^{11}\,\text{N/m}^2$, Stabdurchmesser
$d = 1\,\text{cm}$.

Skizzieren Sie das Fachwerk vor dem Rechnen auf Papier. Überprüfen Sie dabei,
welche Stäbe Sie qualitativ auf Zug und welche auf Druck erwarten.

---

## Aufgabe 1: Grundsetup und Visualisierung

Legen Sie die Materialeigenschaften, die Knotenkoordinaten, die Lagerknoten,
die Konnektivitätsmatrix und den Kraftvektor an. Stellen Sie das Fachwerk
anschließend mit der bereitgestellten Funktion `zeichne_fachwerk` grafisch dar
und prüfen Sie, ob die Geometrie Ihrer Skizze entspricht.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Hier Ihren Code eingeben

Die folgende Funktion ist fertig vorgegeben und darf nicht verändert werden.
Sie zeichnet die Ausgangslage sowie, falls übergeben, die verformte Lage und
farblich kodierte Stabkräfte.

In [ ]:
def zeichne_fachwerk(knoten_pos, verbindung, lager_indizes,
                     verschiebung=None, skalierung=500,
                     stabkraefte=None, titel=''):
    """Zeichnet ein Fachwerk in Ausgangs- und verformter Lage.

    Stäbe werden blau (Zug) oder rot (Druck) eingefärbt, sofern ein
    Dictionary ``stabkraefte`` übergeben wird. Ohne dieses Argument werden
    alle Stäbe blau gezeichnet.

    Parameters
    ----------
    knoten_pos : ndarray, Form (n, 2)
        Knotenkoordinaten; Knoten n steht in Zeile n.
    verbindung : ndarray, Form (n, n)
        Symmetrische Konnektivitätsmatrix.
    lager_indizes : list
        Indizes der fest gelagerten Knoten.
    verschiebung : ndarray oder None
        Verschiebungsvektor der Länge 2n. Ohne Angabe: Nullverschiebung.
    skalierung : float
        Überhöhungsfaktor für die Verformungsdarstellung.
    stabkraefte : dict oder None
        {(i, j): Stabkraft in N}. Positiv = Zug (blau), negativ = Druck (rot).
    titel : str
        Diagrammtitel.
    """
    n = knoten_pos.shape[0]
    if verschiebung is None:
        verschiebung = np.zeros(2 * n)
    if stabkraefte is None:
        stabkraefte = {}

    fig, ax = plt.subplots(figsize=(9, 5))
    knoten_verformt = knoten_pos + skalierung * verschiebung.reshape((n, 2))

    for i in range(n):
        for j in range(i + 1, n):
            if verbindung[i, j]:
                ax.plot([knoten_pos[i, 0],      knoten_pos[j, 0]],
                        [knoten_pos[i, 1],      knoten_pos[j, 1]],
                        color='gray', linewidth=1.5, alpha=0.3)
                F = stabkraefte.get((i, j), 0)
                farbe = 'tab:red' if (stabkraefte and F < 0) else 'tab:blue'
                ax.plot([knoten_verformt[i, 0], knoten_verformt[j, 0]],
                        [knoten_verformt[i, 1], knoten_verformt[j, 1]],
                        color=farbe, linewidth=2.5)
                if stabkraefte:
                    mx = 0.5 * (knoten_verformt[i, 0] + knoten_verformt[j, 0])
                    my = 0.5 * (knoten_verformt[i, 1] + knoten_verformt[j, 1])
                    ax.text(mx, my + 0.07, f'{F / 1000:.2f} kN',
                            fontsize=7, ha='center', color=farbe)

    ax.scatter(knoten_pos[:, 0], knoten_pos[:, 1],
               c='gray', s=60, zorder=4, alpha=0.3)
    ax.scatter(knoten_verformt[:, 0], knoten_verformt[:, 1],
               c='tab:red', s=80, zorder=5)
    for k in range(n):
        ax.text(knoten_verformt[k, 0] + 0.04,
                knoten_verformt[k, 1] + 0.04,
                f'K{k}', fontsize=9)

    h_d, b_d = 0.10, 0.10
    for k in lager_indizes:
        xd = [knoten_verformt[k, 0],
              knoten_verformt[k, 0] - b_d / 2,
              knoten_verformt[k, 0] + b_d / 2]
        yd = [knoten_verformt[k, 1],
              knoten_verformt[k, 1] - h_d,
              knoten_verformt[k, 1] - h_d]
        ax.fill(xd, yd, color='tab:green', alpha=0.7)

    if stabkraefte:
        ax.plot([], [], color='tab:blue', linewidth=3, label='Zug')
        ax.plot([], [], color='tab:red',  linewidth=3, label='Druck')
        ax.legend(fontsize=9, loc='upper right')

    ax.set_title(titel)
    ax.set_aspect('equal')
    ax.grid(True)
    plt.tight_layout()
    plt.show()

---

## Aufgabe 2: Steifigkeitsmatrix aufbauen und LGS lösen

Bauen Sie die globale Steifigkeitsmatrix `steifigkeit_global` auf. Bestimmen
Sie die freien Freiheitsgrade, reduzieren Sie das LGS und lösen Sie es mit
`np.linalg.solve`. Geben Sie den vollständigen Verschiebungsvektor knotenweise
aus.

In [ ]:
# Hier Ihren Code eingeben

---

## Aufgabe 3: Lagerkräfte und Gleichgewichtsprüfung

Berechnen Sie den vollständigen Kraftvektor aus
$\vec{F} = \mathbf{K} \cdot \vec{u}$.
An den freien Knoten liefert er die äußeren Lasten zurück (Probe).
An den Lagerknoten liefert er die Lagerreaktionen. Prüfen Sie das
Gleichgewicht mit `np.allclose` und stellen Sie die Ergebnisse knotenweise
dar.

In [ ]:
# Hier Ihren Code eingeben

---

## Aufgabe 4: Stabkräfte und Visualisierung

Berechnen Sie für jeden Stab die Stabkraft aus der Projektion der relativen
Knotenverschiebung auf die Stabachse:

\begin{equation*}
F_{ij} = k \cdot \vec{e}^\top (\vec{u}_j - \vec{u}_i).
\end{equation*}

Positive Stabkraft bedeutet Zug, negative Stabkraft Druck. Stellen Sie
anschließend die verformte Lage mit farblich kodierten Stabkräften dar und
vergleichen Sie mit Ihrer Vorhersage aus der Skizze.

In [ ]:
# Hier Ihren Code eingeben

---

## Aufgabe 5: Vertiefende Aufgaben

**Hinweis zur Notebook-Reihenfolge:**
Arbeiten Sie in den folgenden Aufgaben in neuen Code-Zellen, ohne die
Originalwerte oben zu überschreiben. So bleiben die Referenzergebnisse
erhalten und Sie können die Resultate direkt vergleichen.

### Vertiefung

1. **Maximale Beanspruchung:** Welcher Stab hat den größten Betrag der
   Stabkraft? Ist das qualitativ plausibel, wenn Sie die Laststellung und
   die Geometrie betrachten?

2. **Einfluss des Querschnitts:** Halbieren Sie den Stabdurchmesser auf
   $d = 0{,}5\,\text{cm}$ und berechnen Sie das System neu. Um welchen
   Faktor ändert sich die maximale vertikale Absenkung? Begründen Sie den
   Zusammenhang mithilfe von $k = EA/L$ ohne Code.

3. **Asymmetrische Last:** Behalten Sie nur die Last von $3000\,\text{N}$
   nach unten an Knoten 5 und entfernen Sie die Last an Knoten 4.
   Warum sind die horizontalen Verschiebungen der Obergurtknoten nun
   nicht mehr null? Prüfen Sie Ihre Antwort mit Code.

In [ ]:
# Hier Ihren Code für die Vertiefungsaufgaben eingeben

## Zusammenfassung

In dieser Übung haben Sie das FEM-Schema für ebene Fachwerke ohne
Schritt-für-Schritt-Anleitung auf ein sechsknötiges Dachstuhl-Fachwerk
angewendet. Der Algorithmus aus den Kapiteln 4.1 bis 4.4 bleibt unverändert:
Geometrie beschreiben, Steifigkeitsmatrix assemblieren, LGS reduzieren,
lösen, Lagerkräfte und Stabkräfte zurückrechnen. Die Größe der Matrizen
wächst mit der Knotenzahl, der Code ändert sich nicht.

Im nächsten Kapitel erweitern Sie das Modell um ein neues Konzept: das
Loslager (Rollenlager), das nur einen Freiheitsgrad sperrt. Dazu wird
die bisherige Variable `lager_indizes` durch eine flexiblere Liste
gesperrter Freiheitsgrade ersetzt.